In [ ]:
# 1️⃣ Google Colab yetkilendirme
from google.colab import auth
auth.authenticate_user()

In [ ]:
# 2️⃣ BigQuery client
from google.cloud import bigquery
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time

In [ ]:
PROJECT_ID = "bigquery-test-dilara-475511"
client = bigquery.Client(project=PROJECT_ID, location="US")
print("Aktif proje:", client.project)

Aktif proje: bigquery-test-dilara-475511


In [ ]:
table_id = f"{PROJECT_ID}.realtime_traffic.traffic_data"

# CSV dosyası adı
csv_file = "realtime_traffic_data.csv"

In [ ]:
# Veri üretme fonksiyonu
def generate_traffic_data(num_rows=10):
    now = datetime.now(timezone.utc)
    data = {
        "timestamp": [(now - timedelta(seconds=i*10)) for i in range(num_rows)],
        "location_id": [f"loc_{np.random.randint(1,5)}" for _ in range(num_rows)],
        "vehicle_count": np.random.randint(5, 50, size=num_rows),
        "average_speed": np.random.uniform(20, 120, size=num_rows)
    }
    return pd.DataFrame(data)

In [ ]:
# Sonsuz döngü: durdurulana kadar çalışır
batch = 1
try:
    while True:
        # 1️⃣ Veri üret
        df = generate_traffic_data(10)

        # 2️⃣ BigQuery'ye yükle
        job = client.load_table_from_dataframe(df, table_id)
        job.result()
        print(f"{len(df)} satır yüklendi. (batch {batch})")

        # 3️⃣ Aynı veriyi CSV'ye ekle
        if batch == 1:
            df.to_csv(csv_file, index=False)
        else:
            df.to_csv(csv_file, mode='a', header=False, index=False)

        batch += 1
        time.sleep(5)  # her 5 saniyede bir batch
except KeyboardInterrupt:
    print("⛔ Veri akışı manuel olarak durduruldu.")
    print(f"Tüm batchler CSV olarak kaydedildi: {csv_file}")

10 satır yüklendi. (batch 1)
10 satır yüklendi. (batch 2)
10 satır yüklendi. (batch 3)
10 satır yüklendi. (batch 4)
10 satır yüklendi. (batch 5)
10 satır yüklendi. (batch 6)
10 satır yüklendi. (batch 7)
10 satır yüklendi. (batch 8)
10 satır yüklendi. (batch 9)
10 satır yüklendi. (batch 10)
⛔ Veri akışı manuel olarak durduruldu.
Tüm batchler CSV olarak kaydedildi: realtime_traffic_data.csv
